# Trexquant - Earnings Return Prediction Challenge 2025
---

## Configure the competition dataset
When you start editing this Notebook, you need to properly configure the Notebook’s **Input**.
- On the right-hand side of the Notebook, you will find the **“Input”** panel.
- Click the **“Add Input”** button, search for the keyword **“Earnings Return Prediction Challenge 2025Q4”** to locate our competition,and then click the **“+”** button in the lower-right corner.

This will add the competition dataset to the Notebook’s Inputs, allowing you to run the Notebook and use the dataset for the competition.

---

## GAR Candidate Submission

- Your goal is to implement a solution with a correlation score **`≥ 0.03`**.
- You have up to **7 days** to complete this challenge. 
- You may select **2** final submission to be considered for judging.

---

## Important Notes
- Participants should not intentionally overfit and must avoid using any future information to ensure that their models do not have forward bias or inadvertently use future data.

- Common mistake: Be cautious when applying scaling/normalization methods that use data from the entire time period, as this can lead to overfitting.

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import lightgbm as lgb
from scipy.stats import pearsonr
import xgboost as xgb

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/earnings-return-prediction-challenge-2025-q-4/train.csv
/kaggle/input/competitions/earnings-return-prediction-challenge-2025-q-4/test.csv


# Load Dataset

In [2]:
df = pd.read_csv("/kaggle/input/competitions/earnings-return-prediction-challenge-2025-q-4/train.csv")
test_df = pd.read_csv("/kaggle/input/competitions/earnings-return-prediction-challenge-2025-q-4/test.csv")

df      = df.sort_values("di").reset_index(drop=True)
test_df = test_df.sort_values("di").reset_index(drop=True)

# Make Prediction

In [3]:
feature_cols = [f'f_{i}' for i in range(1, 173)]
cat_cols     = ['industry', 'sector', 'top2000', 'top1000', 'top500']

for c in cat_cols:
    df[c] = df[c].astype('category')
    test_df[c] = test_df[c].astype('category')

unique_dates = sorted(df["di"].unique())
split        = int(0.85 * len(unique_dates))

train_df = df[df["di"].isin(unique_dates[:split])].copy()
val_df   = df[df["di"].isin(unique_dates[split:])].copy()


def add_nan_features(data, cols):
    nan_df = data[cols].isna().astype(np.int8)
    nan_df.columns = [c + "_nan" for c in cols]
    return pd.concat([data, nan_df], axis=1)

train_df = add_nan_features(train_df, feature_cols)
val_df   = add_nan_features(val_df, feature_cols)
test_df  = add_nan_features(test_df, feature_cols)

nan_cols = [c + "_nan" for c in feature_cols]
all_features = feature_cols + cat_cols + nan_cols


train_medians = train_df.groupby("di")[feature_cols].median()

def apply_median_fill(data, medians, f_cols):
    data = data.merge(medians, on="di", how="left", suffixes=("", "_med"))
    for col in f_cols:
        data[col] = data[col].fillna(data[col + "_med"])
    data.drop(columns=[col + "_med" for col in f_cols], inplace=True)
    return data

train_df = apply_median_fill(train_df, train_medians, feature_cols)
val_df   = apply_median_fill(val_df,   train_medians, feature_cols)
test_df  = apply_median_fill(test_df,  train_medians, feature_cols)


global_median = train_df[feature_cols].median()
for col in feature_cols:
    val_df[col]  = val_df[col].fillna(global_median[col])
    test_df[col] = test_df[col].fillna(global_median[col])


print("\n--- PHASE 1 ---")

seeds = [42, 123, 2024]
best_iters_lgb, best_iters_xgb = [], []

val_preds_lgb, val_preds_xgb = [], []

for seed in seeds:

    # LightGBM
    model_lgb = lgb.LGBMRegressor(
        n_estimators=2000, learning_rate=0.05,
        num_leaves=48, subsample=0.85,
        colsample_bytree=0.85, min_child_samples=80,
        random_state=seed, verbose=-1
    )
    model_lgb.fit(
        train_df[all_features], train_df["target"],
        eval_set=[(val_df[all_features], val_df["target"])],
        categorical_feature=cat_cols,
        callbacks=[lgb.early_stopping(50, verbose=False)]
    )
    best_iters_lgb.append(model_lgb.best_iteration_)
    val_preds_lgb.append(model_lgb.predict(val_df[all_features]))

    # XGBoost
    model_xgb = xgb.XGBRegressor(
        n_estimators=2000, learning_rate=0.05,
        max_depth=6, subsample=0.85,
        colsample_bytree=0.85,
        random_state=seed,
        enable_categorical=True,
        tree_method="hist",
        early_stopping_rounds=50
    )
    model_xgb.fit(
        train_df[all_features], train_df["target"],
        eval_set=[(val_df[all_features], val_df["target"])],
        verbose=False
    )
    best_iters_xgb.append(model_xgb.best_iteration)
    val_preds_xgb.append(model_xgb.predict(val_df[all_features]))

    print(f"Seed {seed} | LGB: {best_iters_lgb[-1]} | XGB: {best_iters_xgb[-1]}")


df = add_nan_features(df, feature_cols)
df = apply_median_fill(df, train_medians, feature_cols)

for col in feature_cols:
    df[col] = df[col].fillna(global_median[col])


print("\n--- PHASE 2 ---")

preds_lgb, preds_xgb = [], []

for i, seed in enumerate(seeds):

    # LGB
    final_lgb = lgb.LGBMRegressor(
        n_estimators=best_iters_lgb[i], learning_rate=0.05,
        num_leaves=48, subsample=0.85,
        colsample_bytree=0.85, min_child_samples=80,
        random_state=seed, verbose=-1
    )
    final_lgb.fit(df[all_features], df["target"], categorical_feature=cat_cols)
    preds_lgb.append(final_lgb.predict(test_df[all_features]))

    # XGB
    final_xgb = xgb.XGBRegressor(
        n_estimators=best_iters_xgb[i], learning_rate=0.05,
        max_depth=6, subsample=0.85,
        colsample_bytree=0.85,
        random_state=seed,
        enable_categorical=True,
        tree_method="hist"
    )
    final_xgb.fit(df[all_features], df["target"])
    preds_xgb.append(final_xgb.predict(test_df[all_features]))

    print(f"Seed {seed} complete.")


final_pred_lgb = np.mean(preds_lgb, axis=0)
final_pred_xgb = np.mean(preds_xgb, axis=0)


final_pred = 0.6 * final_pred_lgb + 0.4 * final_pred_xgb


--- PHASE 1 ---
Seed 42 | LGB: 82 | XGB: 46
Seed 123 | LGB: 31 | XGB: 17
Seed 2024 | LGB: 43 | XGB: 63

--- PHASE 2 ---
Seed 42 complete.
Seed 123 complete.
Seed 2024 complete.


# Submission
**Requirements:**
- Must contain `"target"` column.
- No `nan` or `inf` in `"target"`.
- At least `10%` of target values must be non-zero.

In [4]:
submission = pd.DataFrame({
    "id": test_df["id"],
    "target": final_pred
})

submission.to_csv("/kaggle/working/submission.csv", index=False)
